# Notebook para extraer los precios de despacho y posdespacho (SPOT)

El AMMM tiene esta info en los excels en las hoja "POE" y la matriz de información hasta el 2026 esta en los rangos: C6:F30

Los links son de la forma:
Para SPOT posdespacho:
"https://www.amm.org.gt/pdfs2/post_despacho/POSDESPACHO_DIARIO/2026/05_MAYO/PD20260518.zip"

Para SPOT despacho:
"https://www.amm.org.gt/pdfs2/programas_despacho/01_PROGRAMAS_DE_DESPACHO_DIARIO/2026/01_PROGRAMAS_DE_DESPACHO_DIARIO/05_MAYO/WEB18052026.xlsx"

generalizando:

f"https://www.amm.org.gt/pdfs2/post_despacho/POSDESPACHO_DIARIO/{año}/{mm_mmmm}/PD{yyyymmdd}.zip"

y

f"https://www.amm.org.gt/pdfs2/programas_despacho/01_PROGRAMAS_DE_DESPACHO_DIARIO/{año}/01_PROGRAMAS_DE_DESPACHO_DIARIO/{mm_mmmm}/WEB{ddmmyyyy}.xlsx"

In [ ]:
# Librerias necesarias - Python 3.14.3
import requests
import zipfile
import io
import pandas as pd
import datetime as dt

Script un solo día

In [ ]:
# Función para obtener el postdespacho y despacho
def get_spot(url_posdespacho, url_despacho):
    # --- 1. POSTDESPACHO (Archivo .zip) ---
    response = requests.get(url_posdespacho) # Se pide la respuesta del url (archivo .zip)
    response.raise_for_status() # Respuesta si existe error al descargar 

    with zipfile.ZipFile(io.BytesIO(response.content)) as thezip: # leemos los bits del zip
        # Obtener la lista de archivos dentro del ZIP
        archivos_en_zip = thezip.namelist()
        
        # Validación de seguridad: asegurar que el ZIP no esté vacío
        if not archivos_en_zip:
            raise ValueError("El archivo ZIP descargado está vacío.")
            
        # Filtrar para obtener solo el archivo .xlsx 
        archivos_excel = [f for f in archivos_en_zip if f.endswith('.xlsx')] # en el zip solo existe un xlsx
        
        # Captura de error
        if not archivos_excel:
            raise ValueError("No se encontró ningún archivo .xlsx en el ZIP.")
            
        nombre_excel = archivos_excel[0] # primer archivo dentro del zip (Esperamos solo uno)

        # Abrir el archivo encontrado
        with thezip.open(nombre_excel) as f: # lectura del excel 
            # Convertir a Data Frame
            df_posdespacho = pd.read_excel(
                f, # La data del excel
                sheet_name="POE", # Nombre de la hoja "POE" en el caso del SPOT
                usecols="C:F",    # Delimita las columnas desde la C hasta la F
                skiprows=5,       # Omite las primeras 5 filas (inicia la lectura en la fila 6)
                nrows=24          # Lee 24 filas a partir de la fila 6 (concluye en la fila 30)
            )

    # --- 2. DESPACHO (Archivo .xlsx directo) ---
    response_desp = requests.get(url_despacho) # Se pide la respuesta del segundo url
    response_desp.raise_for_status() 

    # Como es un Excel directo, se lee con io.BytesIO sin necesidad de extraerlo
    df_despacho = pd.read_excel(
        io.BytesIO(response_desp.content), 
        sheet_name="POE", 
        usecols="C:F",    
        skiprows=8,       # Omite las primeras 8 filas (inicia la lectura en la fila 9)
        nrows=24          # Lee 24 filas a partir de la fila 9 (concluye en la fila 33)
    )

    return df_despacho, df_posdespacho # regresa dos dfs despacho y posdespacho


# Función para extraer el SPOT de solo un día, devuelve un dataframe 
def extract(anio: int ,mes: int ,dia: int) -> pd.DataFrame: 
    """
    Procesa la fecha seleccionada, consulta al AMM, formatea y crea un DataFrame con el precio de SPOT del día.

    ### Args:
        **anio** (int): El año correspondiente (ej. 2026).
        **mes** (int): El mes del año, representado del 1 al 12.
        **dia** (int): El día del mes, representado del 1 al 31.

    ### Returns:
        **DataFrame**: objeto DataFrame de pandas.
    """

    # Lista estática para garantizar el mes en español y en mayúsculas
    MESES = [
        "", "ENERO", "FEBRERO", "MARZO", "ABRIL", "MAYO", "JUNIO",
        "JULIO", "AGOSTO", "SEPTIEMBRE", "OCTUBRE", "NOVIEMBRE", "DICIEMBRE"
    ]

    print("Procesando....")
    yymmdd = dt.date(anio,mes,dia).strftime("%Y%m%d") # Formatear fecha
    ddmmyy = dt.date(anio,mes,dia).strftime("%d%m%Y") # Formatear fecha
    mm = dt.date(anio,mes,dia).strftime("%m") # mes en número
    nombre_mes = MESES[mes]  # Mes dado el número del mes dada la lista MESES con MESES[0] = " " para sincronizar la iteración
    mm_mmmm = f"{mm}_{nombre_mes}" # Juntando todo
    urlpd = f"https://www.amm.org.gt/pdfs2/post_despacho/POSDESPACHO_DIARIO/{anio}/{mm_mmmm}/PD{yymmdd}.zip" # creación del url posdespacho
    urld = f"https://www.amm.org.gt/pdfs2/programas_despacho/01_PROGRAMAS_DE_DESPACHO_DIARIO/{anio}/01_PROGRAMAS_DE_DESPACHO_DIARIO/{mm_mmmm}/WEB{ddmmyy}.xlsx"

    try:
        dfd, dfpd = get_spot(url_despacho=urld,url_posdespacho=urlpd) # llamada a función para extraer los dfs del AMM
        # Formateo del DF posdespacho ------------------------------------------------
        dfpd.drop(columns="A",inplace=True)
        dfpd.columns = ["hora","generadora_posdespacho","precio_spot_posdespacho"]
        dfpd["fecha"] = dt.date(anio,mes,dia) 
        dfpd = dfpd[["fecha","hora","precio_spot_posdespacho","generadora_posdespacho"]]
        #-----------------------------------------------------------------------------------
        # Formateo del DF despacho ------------------------------------------------------------------------
        dfd.drop(columns="A:",inplace=True)
        dfd.columns = ["hora","precio_spot_despacho","generadora_despacho"]
        dfd["fecha"] = dt.date(anio,mes,dia) 
        dfd = dfd[["fecha","hora","precio_spot_despacho","generadora_despacho"]]
        #-------------------------------------------------------------------------------
        # Unión final y formato de tabla para carga-------------------
        df_final = pd.merge(dfd, dfpd, on=["fecha", "hora"])
        # Añadir las columnas extra
        columnas_adicionales = [
            "precio_carbon", 
            "precio_bunker", 
            "precio_referencia_ite", 
            "precio_demanda_ite", 
            "tipo_cambio"
        ]
        for col in columnas_adicionales:
            df_final[col] = 0 # seteamos sus valores a 0
        #Reordenar las columnas para garantizar el formato exacto solicitado
        orden_columnas = [
            "fecha", 
            "hora", 
            "precio_spot_despacho", 
            "generadora_despacho", 
            "precio_spot_posdespacho", 
            "generadora_posdespacho", 
            "precio_carbon", 
            "precio_bunker", 
            "precio_referencia_ite", 
            "precio_demanda_ite", 
            "tipo_cambio"
        ]
        df_final = df_final[orden_columnas]
        #------------------------------------------------------
        print(f"Df {ddmmyy} extraido!")
        return df_final
    except Exception as e:
        print(f"Error extrayendo {ddmmyy}, code:{e}")


Script varios días

In [ ]:
# librerias para script de extracción masiva
import os
import calendar
import datetime as dt
import pandas as pd
import concurrent.futures

In [ ]:
# Lista estática para garantizar el mes en español y en mayúsculas
MESES = [
    "", "ENERO", "FEBRERO", "MARZO", "ABRIL", "MAYO", "JUNIO",
    "JULIO", "AGOSTO", "SEPTIEMBRE", "OCTUBRE", "NOVIEMBRE", "DICIEMBRE"
]

# --- CONFIGURACIÓN DE RUTAS Y CHECKPOINTS ---
CARPETA_CHECKPOINTS = "datos_mensuales"
os.makedirs(CARPETA_CHECKPOINTS, exist_ok=True)

years = range(2021, 2027)  # Rango de años a adquirir: 2021 a 2026
months = range(1, 13)      # Rango de meses del 1 al 12

MESES = [
    "", "ENERO", "FEBRERO", "MARZO", "ABRIL", "MAYO", "JUNIO",
    "JULIO", "AGOSTO", "SEPTIEMBRE", "OCTUBRE", "NOVIEMBRE", "DICIEMBRE"
]

# Definición dinámica de la fecha límite (día de ayer)
hoy = dt.date.today()
fecha_limite = hoy - dt.timedelta(days=2)

MAX_WORKERS = 20  # Cantidad de descargas simultáneas (ajustar según capacidad de red)

# --- FUNCIÓN DE EXTRACCIÓN POR DÍA (WORKER) ---
def procesar_dia(fecha_iteracion):
    """Descarga y formatea la información de un solo día."""
    i = fecha_iteracion.year
    mm = fecha_iteracion.strftime("%m")
    nombre_mes = MESES[fecha_iteracion.month]
    
    yymmdd = fecha_iteracion.strftime("%Y%m%d")
    ddmmyy = fecha_iteracion.strftime("%d%m%Y")
    mm_mmmm = f"{mm}_{nombre_mes}"
    
    urlpd = f"https://www.amm.org.gt/pdfs2/post_despacho/POSDESPACHO_DIARIO/{i}/{mm_mmmm}/PD{yymmdd}.zip"
    urld = f"https://www.amm.org.gt/pdfs2/programas_despacho/01_PROGRAMAS_DE_DESPACHO_DIARIO/{i}/01_PROGRAMAS_DE_DESPACHO_DIARIO/{mm_mmmm}/WEB{ddmmyy}.xlsx"

    # Se asume que get_spot está definida previamente en su código
    dfd, dfpd = get_spot(url_despacho=urld, url_posdespacho=urlpd)
    
    # Formateo del DF posdespacho
    dfpd.drop(columns="A", inplace=True)
    dfpd.columns = ["hora", "generadora_posdespacho", "precio_spot_posdespacho"]
    dfpd["fecha"] = fecha_iteracion
    dfpd = dfpd[["fecha", "hora", "precio_spot_posdespacho", "generadora_posdespacho"]]
    
    # Formateo del DF despacho
    dfd.drop(columns="A:", inplace=True)
    dfd.columns = ["hora", "precio_spot_despacho", "generadora_despacho"]
    dfd["fecha"] = fecha_iteracion
    dfd = dfd[["fecha", "hora", "precio_spot_despacho", "generadora_despacho"]]
    
    # Unión final y formato de tabla
    df_final = pd.merge(dfd, dfpd, on=["fecha", "hora"])
    
    columnas_adicionales = [
        "precio_carbon", "precio_bunker", "precio_referencia_ite", 
        "precio_demanda_ite", "tipo_cambio"
    ]
    for col in columnas_adicionales:
        df_final[col] = 0
        
    orden_columnas = [
        "fecha", "hora", "precio_spot_despacho", "generadora_despacho", 
        "precio_spot_posdespacho", "generadora_posdespacho", "precio_carbon", 
        "precio_bunker", "precio_referencia_ite", "precio_demanda_ite", "tipo_cambio"
    ]
    df_final = df_final[orden_columnas]
    return df_final, ddmmyy


# --- BUCLE PRINCIPAL ---
dfs = []  

print("Iniciando proceso con puntos de guardado mensuales y multithreading...")

for i in years:
    for j in months:
        mm = str(j).zfill(2)
        nombre_mes = MESES[j]
        archivo_mes = os.path.join(CARPETA_CHECKPOINTS, f"spot_{i}_{mm}_{nombre_mes}.csv")
        
        es_mes_pasado = (i < hoy.year) or (i == hoy.year and j < hoy.month)
        
        # Validación de checkpoint
        if es_mes_pasado and os.path.exists(archivo_mes):
            print(f"Checkpoint detectado para {nombre_mes} {i}. Cargando datos locales...")
            try:
                df_mes_guardado = pd.read_csv(archivo_mes, parse_dates=["fecha"])
                # Convertir fechas a dt.date para mantener consistencia
                df_mes_guardado["fecha"] = pd.to_datetime(df_mes_guardado["fecha"]).dt.date 
                dfs.append(df_mes_guardado)
                continue
            except Exception as e:
                print(f"Error al leer el archivo {archivo_mes}. Se procesará de nuevo. Detalle: {e}")

        # Generar la lista de fechas válidas a descargar en el mes
        _, num_days = calendar.monthrange(i, j)
        fechas_a_procesar = []
        for k in range(1, num_days + 1):
            fecha_iter = dt.date(i, j, k)
            if fecha_iter <= fecha_limite:
                fechas_a_procesar.append(fecha_iter)

        if not fechas_a_procesar:
            continue

        dfs_mes = []

        print(f"Descargando {len(fechas_a_procesar)} días de {nombre_mes} {i} en paralelo...")

        # Ejecución paralela
        with concurrent.futures.ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
            futuros = {executor.submit(procesar_dia, f): f for f in fechas_a_procesar}
            
            for futuro in concurrent.futures.as_completed(futuros):
                fecha_obj = futuros[futuro]
                try:
                    df_resultado, ddmmyy = futuro.result()
                    dfs_mes.append(df_resultado)
                    print(f"Df {ddmmyy} extraído!")
                except Exception as e:
                    # Log del error; el bucle omite este día y continúa con el resto
                    print(f"[LOG] Omisión en {fecha_obj.strftime('%d/%m/%Y')}: Archivo no disponible o error. Detalle: {e}")

        # Guardado del save point mensual
        if dfs_mes:
            df_mes_unificado = pd.concat(dfs_mes, ignore_index=True)
            # Ordenar para revertir el desorden causado por la asincronía
            df_mes_unificado.sort_values(by=["fecha", "hora"], inplace=True)
            df_mes_unificado.to_csv(archivo_mes, index=False)
            dfs.append(df_mes_unificado)
            print(f"--- Guardado exitoso del checkpoint mensual: {archivo_mes} ---")
        else:
            print(f"--- Aviso: No se extrajeron datos para {nombre_mes} {i}. Se continuará con el siguiente mes. ---")

# --- UNIFICACIÓN FINAL ---
if dfs:
    spot = pd.concat(dfs, ignore_index=True)
    spot.to_csv("spot.csv", index=False)
    print("Dataframes unificados y archivo 'spot.csv' generado correctamente.")
else:
    print("No se encontraron datos para consolidar.")

Iniciando proceso con puntos de guardado mensuales y multithreading...
Checkpoint detectado para ENERO 2021. Cargando datos locales...
Checkpoint detectado para FEBRERO 2021. Cargando datos locales...
Checkpoint detectado para MARZO 2021. Cargando datos locales...
Checkpoint detectado para ABRIL 2021. Cargando datos locales...
Checkpoint detectado para MAYO 2021. Cargando datos locales...
Checkpoint detectado para JUNIO 2021. Cargando datos locales...
Checkpoint detectado para JULIO 2021. Cargando datos locales...
Checkpoint detectado para AGOSTO 2021. Cargando datos locales...
Checkpoint detectado para SEPTIEMBRE 2021. Cargando datos locales...
Checkpoint detectado para OCTUBRE 2021. Cargando datos locales...
Checkpoint detectado para NOVIEMBRE 2021. Cargando datos locales...
Checkpoint detectado para DICIEMBRE 2021. Cargando datos locales...
Checkpoint detectado para ENERO 2022. Cargando datos locales...
Checkpoint detectado para FEBRERO 2022. Cargando datos locales...
Checkpoint det

In [21]:
pd.read_csv("spot.csv")

,fecha,hora,precio_spot_despacho,generadora_despacho,precio_spot_posdespacho,generadora_posdespacho,precio_carbon,precio_bunker,precio_referencia_ite,precio_demanda_ite,tipo_cambio
0,2021-01-01,00:00:00,12.088217,RENACE,17.799713,SANTA ANA,0,0,0,0,0
1,2021-01-01,01:00:00,12.365143,EL MANANTIAL 3,18.283669,HIDROELÉCTRICA GUAYACÁN,0,0,0,0,0
2,2021-01-01,02:00:00,12.365143,EL MANANTIAL 3,18.283669,HIDROELÉCTRICA GUAYACÁN,0,0,0,0,0
3,2021-01-01,03:00:00,12.365143,EL MANANTIAL 3,18.283669,HIDROELÉCTRICA GUAYACÁN,0,0,0,0,0
4,2021-01-01,04:00:00,12.365143,EL MANANTIAL 3,18.283669,HIDROELÉCTRICA GUAYACÁN,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...
47155,2026-05-20,19:00:00,211.029217,"TÉRMICA B3,B4",225.244620,GENOSA,0,0,0,0,0
47156,2026-05-20,20:00:00,211.029217,"TÉRMICA B3,B4",225.244620,GENOSA,0,0,0,0,0
47157,2026-05-20,21:00:00,205.931719,"TDL U3, U4, U9",224.354559,GENOSA,0,0,0,0,0
47158,2026-05-20,22:00:00,200.183563,CHIXOY,224.354559,GENOSA,0,0,0,0,0


Para comprobar que se descargaron todas las fechas

In [ ]:
df = pd.read_csv("spot.csv")

# 1. Nombre de la columna a checar
columna_fecha = "fecha" 

# 2. Convertir la columna a formato datetime
# Si su función formatear_df guardó las fechas como 'DD/MM/YYYY', use dayfirst=True
df[columna_fecha] = pd.to_datetime(df[columna_fecha], format='%Y-%m-%d', errors='coerce')

# 3. Extraer las fechas únicas encontradas en el CSV (ignorando la hora si la tuviera)
fechas_extraidas = pd.to_datetime(df[columna_fecha].dt.date.dropna().unique())

# 4. Generar el rango de fechas esperado
fechas_esperadas = pd.date_range(start="2021-01-01", end="2026-05-21")

# 5. Calcular la diferencia matemática (Fechas Esperadas - Fechas Extraídas)
fechas_faltantes = fechas_esperadas.difference(fechas_extraidas)

# 6. Mostrar los resultados
if len(fechas_faltantes) == 0:
    print("Validación completada: No falta ninguna fecha en el rango especificado.")
else:
    print(f"Alerta: Faltan {len(fechas_faltantes)} fechas en el CSV.")
    print("Lista de fechas faltantes:")
    for fecha in fechas_faltantes:
        print(fecha.date())

Alerta: Faltan 2 fechas en el CSV.
Lista de fechas faltantes:
2026-02-13
2026-05-21


# Pruebas de la función y de formateo de tabla

In [ ]:
dfd, dfpd = get_spot(url_despacho="https://www.amm.org.gt/pdfs2/programas_despacho/01_PROGRAMAS_DE_DESPACHO_DIARIO/2021/01_PROGRAMAS_DE_DESPACHO_DIARIO/01_ENERO/WEB01012021.xlsx",url_posdespacho=f"https://www.amm.org.gt/pdfs2/post_despacho/POSDESPACHO_DIARIO/{2026}/05_MAYO/PD20260518.zip")

In [56]:
dfd.dtypes

DE:                    object
A:                     object
(US $/MWH)            float64
CENTRAL GENERADORA        str
dtype: object

In [57]:
dfpd.dtypes

De                     object
A                      object
GENERADOR MARGINAL        str
POE (US$/MWh)         float64
dtype: object

In [74]:
dfpd.drop(columns="A",inplace=True)
dfpd.columns = ["hora","generadora_posdespacho","precio_spot_posdespacho"]
dfpd["fecha"] = dt.date(2026,5,18) 
dfpd = dfpd[["fecha","hora","precio_spot_posdespacho","generadora_posdespacho"]]

In [75]:
dfd.drop(columns="A:",inplace=True)
dfd.columns = ["hora","precio_spot_despacho","generadora_despacho"]
dfd["fecha"] = dt.date(2026,5,18) 
dfd = dfd[["fecha","hora","precio_spot_despacho","generadora_despacho"]]

In [78]:
df_final = pd.merge(dfd, dfpd, on=["fecha", "hora"])

# 2. Añadir las columnas de variables macroeconómicas e ITEs inicializadas en 0
columnas_adicionales = [
    "precio_carbon", 
    "precio_bunker", 
    "precio_referencia_ite", 
    "precio_demanda_ite", 
    "tipo_cambio"
]

for col in columnas_adicionales:
    df_final[col] = 0

# 3. Reordenar las columnas para garantizar el formato exacto solicitado
orden_columnas = [
    "fecha", 
    "hora", 
    "precio_spot_despacho", 
    "generadora_despacho", 
    "precio_spot_posdespacho", 
    "generadora_posdespacho", 
    "precio_carbon", 
    "precio_bunker", 
    "precio_referencia_ite", 
    "precio_demanda_ite", 
    "tipo_cambio"
]

df_final = df_final[orden_columnas]

In [79]:
df_final

,fecha,hora,precio_spot_despacho,generadora_despacho,precio_spot_posdespacho,generadora_posdespacho,precio_carbon,precio_bunker,precio_referencia_ite,precio_demanda_ite,tipo_cambio
0,2026-05-18,00:00:00,12.088217,RENACE,165.939501,GENOR,0,0,0,0,0
1,2026-05-18,01:00:00,12.365143,EL MANANTIAL 3,151.196194,LAS PALMAS 5,0,0,0,0,0
2,2026-05-18,02:00:00,12.365143,EL MANANTIAL 3,149.690374,ARIZONA,0,0,0,0,0
3,2026-05-18,03:00:00,12.365143,EL MANANTIAL 3,118.121699,SAN ISIDRO,0,0,0,0,0
4,2026-05-18,04:00:00,12.365143,EL MANANTIAL 3,118.121699,SAN ISIDRO,0,0,0,0,0
5,2026-05-18,05:00:00,12.365143,EL MANANTIAL 3,165.939501,GENOR,0,0,0,0,0
6,2026-05-18,06:00:00,12.408317,EL MANANTIAL 3,193.666094,TRINIDAD BLOQUE 3,0,0,0,0,0
7,2026-05-18,07:00:00,12.978700,HIDROELÉCTRICA KAPLAN CHAPINA,193.666094,TRINIDAD BLOQUE 3,0,0,0,0,0
8,2026-05-18,08:00:00,12.978700,HIDROELÉCTRICA KAPLAN CHAPINA,198.087016,TEXTILES DEL LAGO 2,0,0,0,0,0
9,2026-05-18,09:00:00,12.978700,HIDROELÉCTRICA KAPLAN CHAPINA,198.087016,TEXTILES DEL LAGO 2,0,0,0,0,0
